In [1]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
import re
import emoji

In [2]:
Device = 'cuda' if torch.cuda.is_available() else 'cpu'
Device

'cuda'

In [3]:
DF = pd.read_csv('Dataset/english_comments.csv')
DF.head()

,CommentText,VideoTitle,Sentiment
0,Sensationalism. Have they forgotten Yugoslavia...,Russia's invasion of Ukraine: War has returned...,Negative
1,Please rullers be calm\nTalk your self\nIt nev...,Russia attacks Ukraine: Latest developments | ...,Positive
2,What a sad day,Bidens welcome Trumps back to the White House,Negative
3,is it bad i did that last month or smt 💀,Oldest trick in the book,Negative
4,You mean the year 2050,Think Fast. Talk Smart | Matt Abrahams | TEDxM...,Neutral


In [4]:
DF = DF[DF['Sentiment'] != 'Neutral']
DF.head()

,CommentText,VideoTitle,Sentiment
0,Sensationalism. Have they forgotten Yugoslavia...,Russia's invasion of Ukraine: War has returned...,Negative
1,Please rullers be calm\nTalk your self\nIt nev...,Russia attacks Ukraine: Latest developments | ...,Positive
2,What a sad day,Bidens welcome Trumps back to the White House,Negative
3,is it bad i did that last month or smt 💀,Oldest trick in the book,Negative
5,tbh I don't really understand why people think...,Top 6 Coding Interview Concepts (Data Structur...,Negative


In [5]:
len(DF)

60000

In [6]:
def clean_text(text: str) -> str:
    text = emoji.demojize(text, delimiters=(" ", " "))
    text = text.replace("_", " ")
    text = re.sub(r"<[^>]+>", " ", text)        
    text = re.sub(r"\S+@\S+\.\S+", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"#(\w+)", r" \1", text)
    text = re.sub(r"@\w+", " ", text)
    text = text.replace("|", " ").replace("/", " ").replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()     
    return text                                  

In [7]:
DF['CommentCleand'] = DF['CommentText'].apply(clean_text)
DF['VideoCleaned'] = DF['VideoTitle'].apply(clean_text)
DF.head()

,CommentText,VideoTitle,Sentiment,CommentCleand,VideoCleaned
0,Sensationalism. Have they forgotten Yugoslavia...,Russia's invasion of Ukraine: War has returned...,Negative,Sensationalism. Have they forgotten Yugoslavia...,Russia's invasion of Ukraine: War has returned...
1,Please rullers be calm\nTalk your self\nIt nev...,Russia attacks Ukraine: Latest developments | ...,Positive,Please rullers be calm Talk your self It never...,Russia attacks Ukraine: Latest developments DW...
2,What a sad day,Bidens welcome Trumps back to the White House,Negative,What a sad day,Bidens welcome Trumps back to the White House
3,is it bad i did that last month or smt 💀,Oldest trick in the book,Negative,is it bad i did that last month or smt skull,Oldest trick in the book
5,tbh I don't really understand why people think...,Top 6 Coding Interview Concepts (Data Structur...,Negative,tbh I don't really understand why people think...,Top 6 Coding Interview Concepts (Data Structur...


In [8]:
len(DF[DF['CommentCleand'] == ''])

0

In [9]:
len(DF[DF['VideoCleaned'] == ''])

0

In [10]:
len(DF[DF['Sentiment'] == ''])

0

In [11]:
DF.isna().sum()

CommentText      0
VideoTitle       0
Sentiment        0
CommentCleand    0
VideoCleaned     0
dtype: int64

In [12]:
DF['FullText'] = DF['VideoCleaned']+' [SEP] '+DF['CommentCleand']
DF.head()

,CommentText,VideoTitle,Sentiment,CommentCleand,VideoCleaned,FullText
0,Sensationalism. Have they forgotten Yugoslavia...,Russia's invasion of Ukraine: War has returned...,Negative,Sensationalism. Have they forgotten Yugoslavia...,Russia's invasion of Ukraine: War has returned...,Russia's invasion of Ukraine: War has returned...
1,Please rullers be calm\nTalk your self\nIt nev...,Russia attacks Ukraine: Latest developments | ...,Positive,Please rullers be calm Talk your self It never...,Russia attacks Ukraine: Latest developments DW...,Russia attacks Ukraine: Latest developments DW...
2,What a sad day,Bidens welcome Trumps back to the White House,Negative,What a sad day,Bidens welcome Trumps back to the White House,Bidens welcome Trumps back to the White House ...
3,is it bad i did that last month or smt 💀,Oldest trick in the book,Negative,is it bad i did that last month or smt skull,Oldest trick in the book,Oldest trick in the book [SEP] is it bad i did...
5,tbh I don't really understand why people think...,Top 6 Coding Interview Concepts (Data Structur...,Negative,tbh I don't really understand why people think...,Top 6 Coding Interview Concepts (Data Structur...,Top 6 Coding Interview Concepts (Data Structur...


In [13]:
label_mapping = {'Negative': 0, 'Positive': 1}
DF['label'] = DF['Sentiment'].map(label_mapping).astype(int)
DF.head()

,CommentText,VideoTitle,Sentiment,CommentCleand,VideoCleaned,FullText,label
0,Sensationalism. Have they forgotten Yugoslavia...,Russia's invasion of Ukraine: War has returned...,Negative,Sensationalism. Have they forgotten Yugoslavia...,Russia's invasion of Ukraine: War has returned...,Russia's invasion of Ukraine: War has returned...,0
1,Please rullers be calm\nTalk your self\nIt nev...,Russia attacks Ukraine: Latest developments | ...,Positive,Please rullers be calm Talk your self It never...,Russia attacks Ukraine: Latest developments DW...,Russia attacks Ukraine: Latest developments DW...,1
2,What a sad day,Bidens welcome Trumps back to the White House,Negative,What a sad day,Bidens welcome Trumps back to the White House,Bidens welcome Trumps back to the White House ...,0
3,is it bad i did that last month or smt 💀,Oldest trick in the book,Negative,is it bad i did that last month or smt skull,Oldest trick in the book,Oldest trick in the book [SEP] is it bad i did...,0
5,tbh I don't really understand why people think...,Top 6 Coding Interview Concepts (Data Structur...,Negative,tbh I don't really understand why people think...,Top 6 Coding Interview Concepts (Data Structur...,Top 6 Coding Interview Concepts (Data Structur...,0


In [14]:
TrSet, TestSet = train_test_split(DF, random_state=42, stratify=DF['label'], test_size=0.1)
TrainSet, valSet = train_test_split(TrSet, random_state=42, stratify=TrSet['label'], test_size=0.1)
TrainSet.head()

,CommentText,VideoTitle,Sentiment,CommentCleand,VideoCleaned,FullText,label
11934,"""it's just paper, but it's fun"" - I HAVE SERIO...",She could be politician😂 #motivationalquotes #...,Negative,"""it's just paper, but it's fun"" - I HAVE SERIO...",She could be politician face with tears of joy...,She could be politician face with tears of joy...,0
15668,Enjoy your day Lakhneet ❤\n🧿Kisiki nazar na lg...,We are in Istanbul 🇹🇷 Friends Trip Like Never ...,Positive,Enjoy your day Lakhneet red heart nazar amulet...,We are in Istanbul Türkiye Friends Trip Like N...,We are in Istanbul Türkiye Friends Trip Like N...,1
8903,"Not good at all, bro",Welcome to Noxus - Bite Marks (ft. TEYA) | 202...,Negative,"Not good at all, bro",Welcome to Noxus - Bite Marks (ft. TEYA) 2025 ...,Welcome to Noxus - Bite Marks (ft. TEYA) 2025 ...,0
8981,Wyoming Oil drilling in the late 18th century ...,First images of volcano damage show Tonga area...,Negative,Wyoming Oil drilling in the late 18th century ...,First images of volcano damage show Tonga area...,First images of volcano damage show Tonga area...,0
40957,FaceBook has a Dark Pattern once i tried to de...,Did you know why apps do this?,Negative,FaceBook has a Dark Pattern once i tried to de...,Did you know why apps do this?,Did you know why apps do this? [SEP] FaceBook ...,0


In [15]:
TrainSet.to_csv('./PreprocecedDatasets/TrainSet.csv')
valSet.to_csv('./PreprocecedDatasets/valSet.csv')
TestSet.to_csv('./PreprocecedDatasets/TestSet.csv')